# Colab starter — Segmentación de heridas en ratas

Correr las celdas en orden. Este notebook clona el repo de GitHub, monta Google Drive (donde están `Images/` y `Masks/`), instala dependencias, y deja todo listo para llamar a los scripts de `src/`.

**Antes de correr:** en Colab, andá a `Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU` para tener GPU disponible.

In [ ]:
# 1) Clonar el repo (reemplazar por tu URL real una vez que lo subas a GitHub)
REPO_URL = "https://github.com/camichahwan/wound_segmentation.git"
!git clone $REPO_URL
%cd wound_segmentation

In [ ]:
# 2) Instalar dependencias
!pip install -q -r requirements.txt

In [ ]:
# 3) Montar Google Drive (te va a pedir autorización la primera vez)
import os, sys

REPO_DIR = "/content/wound_segmentation"
os.chdir(REPO_DIR)  # por si esta celda se corre sola, sin haber corrido la 1 justo antes
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

from config import mount_drive_if_colab, get_data_root, get_images_dir, get_masks_dir, get_checkpoints_dir

mount_drive_if_colab()
print("Data root:", get_data_root())
print("Images:", get_images_dir())
print("Masks:", get_masks_dir())
print("Checkpoints (train.py guarda ACÁ, en Drive -- no en /content, que se borra si se desconecta la sesión):", get_checkpoints_dir())

In [ ]:
# 4) Verificar que el emparejamiento imagen-máscara funciona antes de entrenar nada
from dataset import list_paired_and_unpaired, train_val_test_split

paired, unpaired = list_paired_and_unpaired()
print(f"Imágenes con máscara: {len(paired)}")
print(f"Imágenes sin máscara todavía: {len(unpaired)}")

train_samples, val_samples, test_samples = train_val_test_split(paired)
print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

In [ ]:
# 5) Entrenar la U-Net preentrenada (rápido, buen primer experimento)
!cd src && python train.py --model pretrained --encoder resnet34 --epochs 40 --batch_size 8 --image_size 512

In [ ]:
# 6) Entrenar la U-Net desde cero (más lenta en converger, dejarla correr más épocas)
!cd src && python train.py --model scratch --epochs 80 --batch_size 8 --image_size 512

In [ ]:
# 7) Comparar ambos modelos en el set de test
# Los checkpoints ahora se guardan en Drive (ver celda 3, "Checkpoints"), no en /content,
# así que evaluate.py los busca en esa misma carpeta de Drive.
from config import get_checkpoints_dir

ckpt_dir = get_checkpoints_dir()
pretrained_ckpt = os.path.join(ckpt_dir, "unet_pretrained.pt")
scratch_ckpt = os.path.join(ckpt_dir, "unet_scratch.pt")

!cd src && python evaluate.py --checkpoints "{pretrained_ckpt}" "{scratch_ckpt}"

In [ ]:
# 8) Post-procesamiento + ensemble: ¿ayudan a los errores que NO son de heridas chicas?
# Reutiliza los dos checkpoints ya entrenados (no reentrena nada). Compara contra
# unet_pretrained solo con test pareado de Wilcoxon -- ver outputs/error_analysis/
# despues de correrlo para el reporte con la decision.
!cd src && python postprocess_and_ensemble.py

# Enfoque en dos etapas (heridas chicas/tempranas)

Ver `outputs/error_analysis/heridas_chicas_resize.md`: las heridas más chicas del dataset quedan en ~6x6 píxeles después del resize a 512x512 que usa el pipeline de una sola etapa -- por debajo de lo que la red puede ver. Estas celdas entrenan y evalúan un enfoque en dos etapas: una localización gruesa a mayor resolución, seguida de un recorte + segmentación fina sobre esa región. Correr en orden, cada celda depende de la anterior.

In [ ]:
# 9) Etapa 1: entrenar el localizador a mayor resolución (1024, no 512)
# batch_size mas chico que en la celda 5 porque 1024x1024 pesa mucho mas en memoria de GPU.
# Si tira error de memoria (OOM) en la T4, bajar --batch_size a 2.
!cd src && python train.py --model pretrained --encoder resnet34 --epochs 30 --batch_size 4 \
    --image_size 1024 --run_name unet_localizer_1024

In [ ]:
# 10) Generar el dataset de recortes (a partir de las máscaras reales) para entrenar la etapa 2
!cd src && python make_crop_dataset.py --crop_size 512

In [ ]:
# 11) Etapa 2: fine-tuning de unet_pretrained sobre los recortes (parte de los pesos ya entrenados,
# no de cero -- ver --init_checkpoint). WOUND_DATA_ROOT apunta train.py al dataset de recortes
# generado en la celda anterior, en vez de a las imágenes completas.
import os
from config import get_crops_data_root, get_checkpoints_dir, get_data_root

# OJO: hay que resolver ckpt_dir (y fijarlo con WOUND_CHECKPOINTS_DIR) ANTES
# de pisar WOUND_DATA_ROOT. get_checkpoints_dir() depende de get_data_root(),
# así que si no lo fijamos antes, train.py (que corre como subproceso y
# hereda estas variables de entorno) buscaría unet_pretrained.pt Y guardaría
# el checkpoint nuevo dentro de "Tesis Imagenes Crops/model_checkpoints" en
# vez de "Tesis Imagenes/model_checkpoints", que es donde están los demás
# checkpoints y donde después evaluate_two_stage.py los va a buscar.
ckpt_dir = get_checkpoints_dir()
pretrained_ckpt = os.path.join(ckpt_dir, "unet_pretrained.pt")

os.environ["WOUND_CHECKPOINTS_DIR"] = ckpt_dir
os.environ["WOUND_DATA_ROOT"] = get_crops_data_root()
print("Entrenando etapa 2 sobre:", os.environ["WOUND_DATA_ROOT"])
print("Checkpoints se guardan en:", os.environ["WOUND_CHECKPOINTS_DIR"])
print("Partiendo de checkpoint:", pretrained_ckpt)

!cd src && python train.py --model pretrained --encoder resnet34 --epochs 30 --batch_size 8 \
    --image_size 512 --run_name unet_stage2_finetuned --init_checkpoint "{pretrained_ckpt}"

# IMPORTANTE: sacar los dos overrides apenas termina, para que las celdas siguientes
# (evaluación de punta a punta) vuelvan a usar las imágenes completas originales
# y la carpeta de checkpoints normal, no la de los recortes.
del os.environ["WOUND_DATA_ROOT"]
del os.environ["WOUND_CHECKPOINTS_DIR"]
print("Overrides reseteados. Data root de nuevo:", get_data_root())
print("Checkpoints dir de nuevo:", get_checkpoints_dir())

In [ ]:
# 12) Evaluación de punta a punta: dos etapas vs. unet_pretrained solo, sobre las imágenes
# de test ORIGINALES completas (no recortes) -- esta es la comparación real.
ckpt_dir = get_checkpoints_dir()
stage1_ckpt = os.path.join(ckpt_dir, "unet_localizer_1024.pt")
stage2_ckpt = os.path.join(ckpt_dir, "unet_stage2_finetuned.pt")

!cd src && python evaluate_two_stage.py --stage1_checkpoint "{stage1_ckpt}" --stage2_checkpoint "{stage2_ckpt}"

In [ ]:
# 13) ¿El post-procesamiento arregla las fallas catastroficas nuevas del enfoque en dos etapas?
# Ver claude/analisis_resultados_dos_etapas.md en el proyecto: la celda 12 mejoro Dice/IoU
# de forma clara, pero introdujo 2 fallas catastroficas nuevas (DSC01378, DSC01037) por
# sobre-deteccion de la etapa 1. Esta celda prueba si el mismo post-procesamiento que ya
# funciono bien (celda 8) las arregla. No reentrena nada, es rapida (similar a la celda 8).
ckpt_dir = get_checkpoints_dir()
stage1_ckpt = os.path.join(ckpt_dir, "unet_localizer_1024.pt")
stage2_ckpt = os.path.join(ckpt_dir, "unet_stage2_finetuned.pt")

!cd src && python evaluate_two_stage_postproc.py --stage1_checkpoint "{stage1_ckpt}" --stage2_checkpoint "{stage2_ckpt}"

In [ ]:
# 14) Diagnostico de la etapa 1: ¿por que fallan DSC01378 y DSC01037?
# No cambia ningun modelo: mide que tan grande predice la etapa 1 vs. la herida real y guarda
# imagenes superpuestas (verde=real, rojo=etapa 1, azul=recorte) para VER el problema antes de
# decidir el umbral de la validacion de tamaño. Rapida (solo inferencia).
ckpt_dir = get_checkpoints_dir()
stage1_ckpt = os.path.join(ckpt_dir, "unet_localizer_1024.pt")

!cd src && python diagnose_stage1.py --stage1_checkpoint "{stage1_ckpt}"

In [ ]:
# 15) Dos etapas + filtro en la etapa 1: quedarse solo con la componente mas grande ANTES de calcular el recorte.
# El diagnostico (celda 14) mostro que DSC01378 y DSC01037 fallan porque la etapa 1 trae un blob falso chico
# que infla el recorte, no porque sobre-estime el tamaño. No reentrena nada, es rapida.
ckpt_dir = get_checkpoints_dir()
stage1_ckpt = os.path.join(ckpt_dir, "unet_localizer_1024.pt")
stage2_ckpt = os.path.join(ckpt_dir, "unet_stage2_finetuned.pt")

!cd src && python evaluate_two_stage_s1filter.py --stage1_checkpoint "{stage1_ckpt}" --stage2_checkpoint "{stage2_ckpt}"

# Mejora de precisión con protocolo riguroso (split congelado)

Desde acá las **decisiones** se toman con el split de **validación**; el de **test** se reserva para el número final de la tesis. El split ahora está congelado en `splits/split_v1.csv` (git): sumar imágenes segmentadas nuevas ya no cambia val/test, las nuevas van a train.

Orden: 16 (confirmar el filtro en validación) → 17 (reentrenar etapa 2 pensando en heridas chicas, ~1-2 h) → 18 (comparar en validación).

In [ ]:
# 16) Confirmar en VALIDACION (imagenes que no usamos para diagnosticar) que el filtro de la etapa 1
# funciona. Compara pretrained solo vs dos etapas vs dos etapas + filtro. Rapida, solo inferencia.
ckpt_dir = get_checkpoints_dir()
stage1_ckpt = os.path.join(ckpt_dir, "unet_localizer_1024.pt")
stage2_ckpt = os.path.join(ckpt_dir, "unet_stage2_finetuned.pt")

!cd src && python evaluate_two_stage_s1filter.py --stage1_checkpoint "{stage1_ckpt}" --stage2_checkpoint "{stage2_ckpt}" --split val --tag val_current

In [ ]:
# 17) Reentrenar la etapa 2 pensando en heridas chicas: loss BCE + Focal Tversky (penaliza mas no marcar
# herida que marcar de mas) y muestreo que repite mas las imagenes con herida chica. Parte de unet_pretrained.pt
# (igual que la etapa 2 original) y entrena sobre los recortes. Guarda como unet_stage2_ftv.pt (no pisa nada).
import os
from config import get_crops_data_root, get_checkpoints_dir, get_data_root

# Igual que en la celda 11: fijar los checkpoints ANTES de pisar WOUND_DATA_ROOT.
ckpt_dir = get_checkpoints_dir()
pretrained_ckpt = os.path.join(ckpt_dir, "unet_pretrained.pt")
os.environ["WOUND_CHECKPOINTS_DIR"] = ckpt_dir
os.environ["WOUND_DATA_ROOT"] = get_crops_data_root()
print("Entrenando sobre:", os.environ["WOUND_DATA_ROOT"], "| checkpoints en:", ckpt_dir)

!cd src && python train.py --model pretrained --encoder resnet34 --epochs 40 --batch_size 8 \
    --image_size 512 --run_name unet_stage2_ftv --init_checkpoint "{pretrained_ckpt}" \
    --loss bce_focal_tversky --oversample_small

del os.environ["WOUND_DATA_ROOT"]
del os.environ["WOUND_CHECKPOINTS_DIR"]
print("Overrides reseteados. Data root:", get_data_root())

In [ ]:
# 18) Comparar en VALIDACION la etapa 2 nueva (focal tversky + muestreo) contra la etapa 2 original.
# Mirar sobre todo las 4 heridas chicas y si mejoran Dice/Hausdorff sin perder en el resto.
ckpt_dir = get_checkpoints_dir()
stage1_ckpt = os.path.join(ckpt_dir, "unet_localizer_1024.pt")
stage2_new = os.path.join(ckpt_dir, "unet_stage2_ftv.pt")

!cd src && python evaluate_two_stage_s1filter.py --stage1_checkpoint "{stage1_ckpt}" --stage2_checkpoint "{stage2_new}" --split val --tag val_ftv

In [ ]:
# 19) Diagnostico en VALIDACION de los casos donde dos etapas fallo feo: DSC01102 (el pretrained solo daba 0.91
# y dos etapas 0.00-0.10, area predicha 18x la real), DSC01728 (area predicha 54x) y DSC00952 (Dice 0 en todos).
# Muestra que predijo la etapa 1 (verde=real, rojo=etapa 1, azul=recorte). Rapida, solo inferencia.
ckpt_dir = get_checkpoints_dir()
stage1_ckpt = os.path.join(ckpt_dir, "unet_localizer_1024.pt")

!cd src && python diagnose_stage1.py --stage1_checkpoint "{stage1_ckpt}" --split val --focus DSC01102.JPG DSC01728.JPG DSC00952.JPG